Import Library

In [1]:
# ==========================================
# CELL 1: INITIALIZATION & IMPORTS
# ==========================================
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.losses import Loss
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss

# Bersihkan memori grafik TensorFlow untuk mencegah penumpukan/error dimensi
tf.keras.backend.clear_session()
print("[INFO] Semua library berhasil di-import dan memori TF telah dibersihkan.")

[INFO] Semua library berhasil di-import dan memori TF telah dibersihkan.


Custom Components

In [2]:
# ==========================================
# CELL 2: CUSTOM LOSS & CALLBACK
# ==========================================
class AsymmetricFocalLoss(Loss):
    def __init__(self, gamma_pos=2.0, gamma_neg=4.0, name="asymmetric_focal_loss"):
        super().__init__(name=name)
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg

    def call(self, y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        y_true = tf.cast(y_true, tf.float32)

        bce = -y_true * tf.math.log(y_pred) - (1.0 - y_true) * tf.math.log(1.0 - y_pred)
        weight_pos = tf.pow(1.0 - y_pred, self.gamma_pos)
        weight_neg = tf.pow(y_pred, self.gamma_neg)

        focal_weight = y_true * weight_pos + (1.0 - y_true) * weight_neg
        return tf.reduce_mean(tf.reduce_sum(focal_weight * bce, axis=-1))

class CustomTargetMonitor(Callback):
    def __init__(self, target_val_loss=0.05):
        super().__init__()
        self.target_val_loss = target_val_loss

    def on_epoch_end(self, epoch, logs=None):
        val_loss = logs.get('val_loss')
        if val_loss is not None and val_loss < self.target_val_loss:
            print(f"\n[!] Target Val Loss tercapai ({val_loss:.4f}). Menghentikan training agar model tidak overfit!")
            self.model.stop_training = True

print("[INFO] Custom Components berhasil didefinisikan.")

[INFO] Custom Components berhasil didefinisikan.


Data Loading & Preprocessing Dasar

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
# ==========================================
# CELL 3: DATA PREPARATION & SPLITTING
# ==========================================
file_path = '/content/drive/MyDrive/Capstone/dataset_it_careers_clean.csv'
df = pd.read_csv(file_path)

df['all_skills'] = df['all_skills'].fillna('none').astype(str).str.replace(';', ' ')
df['tools'] = df['tools'].fillna('none').astype(str).str.replace(';', ' ')
df['databases'] = df['databases'].fillna('none').astype(str).str.replace(';', ' ')
df['years_code'] = df['years_code'].fillna(0.0).astype(float)
df['education_level'] = df['education_level'].fillna(0).astype(int)

y_dummies = pd.get_dummies(df['career_label'])
y = y_dummies.astype(float).values
CAREER_NAMES = y_dummies.columns.tolist()
NUM_CAREERS = y.shape[1]

X = {
    'years_code': df['years_code'].to_numpy().reshape(-1, 1),
    'education_level': df['education_level'].to_numpy().reshape(-1, 1),
    # FIX: Teks dibiarkan 1D array murni
    'all_skills': df['all_skills'].to_numpy(),
    'tools': df['tools'].to_numpy(),
    'databases': df['databases'].to_numpy()
}

train_idx, val_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
print(f"[INFO] Data siap! Ditemukan {NUM_CAREERS} klasifikasi karir IT unik.")

[INFO] Data siap! Ditemukan 21 klasifikasi karir IT unik.


Arsitektur Model Deep Learning (Multi Branch)

In [16]:
# ==========================================
# CELL 4: MODEL ARCHITECTURE DEFINITION (TUNED)
# ==========================================
# 1. Definisi Input
input_years = tf.keras.Input(shape=(1,), dtype=tf.float32, name='years_code')
input_edu = tf.keras.Input(shape=(1,), dtype=tf.float32, name='education_level')
input_skills = tf.keras.Input(shape=(), dtype=tf.string, name='all_skills')
input_tools = tf.keras.Input(shape=(), dtype=tf.string, name='tools')
input_databases = tf.keras.Input(shape=(), dtype=tf.string, name='databases')

norm_years = layers.Normalization()
norm_years.adapt(X['years_code'])
processed_years = norm_years(input_years)

norm_edu = layers.Normalization()
norm_edu.adapt(X['education_level'])
processed_edu = norm_edu(input_edu)

MAX_VOCAB = 5000
# TWEAK 1: Menaikkan dimensi embedding agar pemahaman bahasa model lebih kaya
embed_dim = 128

vec_skills = layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=50)
vec_skills.adapt(X['all_skills'])
emb_skills = layers.Embedding(MAX_VOCAB, embed_dim, mask_zero=True)(vec_skills(input_skills))
pool_skills = layers.GlobalAveragePooling1D()(emb_skills)

vec_tools = layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=20)
vec_tools.adapt(X['tools'])
emb_tools = layers.Embedding(MAX_VOCAB, embed_dim, mask_zero=True)(vec_tools(input_tools))
pool_tools = layers.GlobalAveragePooling1D()(emb_tools)

vec_databases = layers.TextVectorization(max_tokens=MAX_VOCAB, output_sequence_length=15)
vec_databases.adapt(X['databases'])
emb_databases = layers.Embedding(MAX_VOCAB, embed_dim, mask_zero=True)(vec_databases(input_databases))
pool_databases = layers.GlobalAveragePooling1D()(emb_databases)

# 2. Penggabungan & Fully Connected Layers
merged = layers.Concatenate()([processed_years, processed_edu, pool_skills, pool_tools, pool_databases])

# TWEAK 2: Memperdalam arsitektur Neural Network (256 -> 128 -> 64)
x = layers.Dense(256, activation='relu')(merged)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x) # Dropout dibesarkan untuk mencegah overfitting pada layer tebal

x = layers.Dense(128, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(64, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)

output_layer = layers.Dense(NUM_CAREERS, activation='sigmoid', name='output')(x)

model = Model(inputs=[input_years, input_edu, input_skills, input_tools, input_databases], outputs=output_layer)

# TWEAK 3: Menurunkan learning rate sedikit agar model belajar lebih hati-hati
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss=AsymmetricFocalLoss(),
    metrics=['accuracy']
)
print("[INFO] Arsitektur V2 (Tuned) berhasil disusun.")

[INFO] Arsitektur V2 (Tuned) berhasil disusun.


Training Pipeline & Export

In [17]:
# ==========================================
# CELL 5: TF PIPELINE, TRAINING & SAVING (TUNED)
# ==========================================
train_ds = tf.data.Dataset.from_tensor_slices(({k: v[train_idx] for k, v in X.items()}, y[train_idx]))
train_ds = train_ds.shuffle(1024).batch(32).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(({k: v[val_idx] for k, v in X.items()}, y[val_idx]))
val_ds = val_ds.batch(32).prefetch(tf.data.AUTOTUNE)

callbacks_list = [
    # Target loss diturunkan sedikit agar model berjuang lebih keras
    CustomTargetMonitor(target_val_loss=0.035),

    # TWEAK 4: Kesabaran (patience) dinaikkan menjadi 8 agar tidak cepat menyerah jika val_loss fluktuatif
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
]

print("\n[INFO] Memulai Proses Training (Extended Epochs)...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    # TWEAK 5: Max epoch dinaikkan ke 50 (biarkan EarlyStopping yang menghentikan di tengah jalan)
    epochs=50,
    callbacks=callbacks_list
)

export_path = 'career_recsys_model.keras'
model.save(export_path)
print(f"\n[SUCCESS] Bobot model V2 berhasil disimpan ke: '{export_path}'")


[INFO] Memulai Proses Training (Extended Epochs)...
Epoch 1/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 15s 16ms/step - accuracy: 0.1857 - loss: 1.6357 - val_accuracy: 0.4459 - val_loss: 0.4044
Epoch 2/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.3936 - loss: 0.4920 - val_accuracy: 0.4842 - val_loss: 0.3650
Epoch 3/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.4492 - loss: 0.4148 - val_accuracy: 0.5072 - val_loss: 0.3460
Epoch 4/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.4859 - loss: 0.3788 - val_accuracy: 0.5272 - val_loss: 0.3322
Epoch 5/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.5101 - loss: 0.3547 - val_accuracy: 0.5353 - val_loss: 0.3214
Epoch 6/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.5232 - loss: 0.3415 - val_accuracy: 0.5363 - val_loss: 0.3160
Epoch 7/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.5332 - loss: 0.3302 - val_accuracy: 0.5423 - val_loss: 0.3123
Epoch 8/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 

Evaluasi Metrik & Baseline

In [18]:
# ==========================================
# CELL 6: EVALUATION METRICS (DL vs ML)
# ==========================================
print("[INFO] Melakukan ekstraksi performa dari Validation Set...")
y_pred_prob = model.predict(val_ds, verbose=0)
y_val_true = y[val_idx]

# Binarisasi probabilitas
y_pred_bin = (y_pred_prob >= 0.5).astype(int)

# Hitung Macro F1 & Hamming Loss
macro_f1 = f1_score(y_val_true, y_pred_bin, average='macro', zero_division=0)
h_loss = hamming_loss(y_val_true, y_pred_bin)

# Hitung Precision@3
def precision_at_k(y_true, y_pred_prob, k=3):
    precisions = []
    for i in range(y_true.shape[0]):
        top_k_idx = np.argsort(y_pred_prob[i])[-k:]
        true_labels = np.where(y_true[i] == 1)[0]
        hits = len(set(top_k_idx) & set(true_labels))
        precisions.append(hits / k)
    return np.mean(precisions)

p_at_3 = precision_at_k(y_val_true, y_pred_prob, k=3)

print("\n" + "="*45)
print("HASIL EVALUASI DEEP LEARNING (VALIDATION SET)")
print("="*45)
print(f"1. Macro F1-Score : {macro_f1:.4f}")
print(f"2. Hamming Loss   : {h_loss:.4f}")
print(f"3. Precision@3    : {p_at_3:.4f}")
print("="*45)

[INFO] Melakukan ekstraksi performa dari Validation Set...

HASIL EVALUASI DEEP LEARNING (VALIDATION SET)
1. Macro F1-Score : 0.1871
2. Hamming Loss   : 0.0436
3. Precision@3    : 0.2858


Backend API Simulation (Inference)

In [19]:
# ==========================================
# CELL 7: INFERENCE SCRIPT (UNTUK BACKEND)
# ==========================================
import tensorflow as tf
import numpy as np

# Load model dengan aman
inference_model = tf.keras.models.load_model('career_recsys_model.keras', compile=False)

def predict_career(user_data, top_k=3):
    # FIX: Gunakan tf.constant secara eksplisit dengan spesifikasi tipe data dan shape yang tepat
    # Numerik membutuhkan shape (1, 1) -> [[nilai]]
    # Teks membutuhkan shape (1,) -> [nilai]
    input_tensor = {
        'years_code': tf.constant([[user_data.get('years_code', 0.0)]], dtype=tf.float32),
        'education_level': tf.constant([[user_data.get('education_level', 0)]], dtype=tf.float32),
        'all_skills': tf.constant([user_data.get('all_skills', 'none')], dtype=tf.string),
        'tools': tf.constant([user_data.get('tools', 'none')], dtype=tf.string),
        'databases': tf.constant([user_data.get('databases', 'none')], dtype=tf.string)
    }

    # Prediksi sekarang akan dijamin aman dari masalah dtype
    probs = inference_model.predict(input_tensor, verbose=0)[0]

    # Ambil index probabilitas tertinggi
    top_indices = np.argsort(probs)[-top_k:][::-1]

    # Format output (API Response)
    recommendations = [
        {"career": CAREER_NAMES[idx], "match_score": float(probs[idx])}
        for idx in top_indices
    ]
    return recommendations

# --- SIMULASI REQUEST API DARI FRONTEND ---
dummy_payload = {
    "years_code": 2.5,
    "education_level": 2,
    "all_skills": "python tensorflow data analysis machine learning",
    "tools": "jupyter notebook google colab git",
    "databases": "mysql sqlite"
}

print("\n[Menerima Data Profil Mahasiswa]:\n", dummy_payload)
print("-" * 45)

hasil = predict_career(dummy_payload)

print(">> REKOMENDASI KARIR (TOP 3):")
for i, rec in enumerate(hasil, 1):
    print(f"{i}. {rec['career']} (Score: {rec['match_score']*100:.1f}%)")


[Menerima Data Profil Mahasiswa]:
 {'years_code': 2.5, 'education_level': 2, 'all_skills': 'python tensorflow data analysis machine learning', 'tools': 'jupyter notebook google colab git', 'databases': 'mysql sqlite'}
---------------------------------------------
>> REKOMENDASI KARIR (TOP 3):
1. Full Stack Developer (Score: 45.6%)
2. Machine Learning Engineer (Score: 44.9%)
3. Data Scientist (Score: 44.6%)
